In [ ]:
import os
import requests
import pandas as pd
from datetime import datetime, timedelta
from dotenv import load_dotenv

load_dotenv()

SERVICE_KEY = os.getenv("SERVICE_KEY")
URL = "https://apis.data.go.kr/B551015/API186_1/SeoulRace_1"

def safe_get_items(data):
    try:
        body = data.get("response", {}).get("body", {})
        items = body.get("items", {})

        if not items:
            return []

        item = items.get("item", None)


        if item is None or item == "":
            return []

        if isinstance(item, dict):
            return [item]

        elif isinstance(item, list):
            return item

        else:
            return []

    except:
        return []

# ----------------------
# API 호출 (page 포함)
# ----------------------
def get_data(start_date, end_date, page):
    params = {
        "ServiceKey": SERVICE_KEY,
        "pageNo": str(page),
        "numOfRows": "1000",   # ⭐ 안정값
        "rc_date_fr": start_date,
        "rc_date_to": end_date,
        "_type": "json"
    }

    response = requests.get(URL, params=params)

    if response.status_code != 200:
        print("요청 실패:", response.status_code)
        return [], 0

    try:
        data = response.json()
        total = data.get("response", {}).get("body", {}).get("totalCount", 0)
    except:
        print("JSON 실패")
        return [], 0

    items = safe_get_items(data)
    return items, total

# ----------------------
# 🔄 날짜 + 페이지 병행
# ----------------------
start = datetime(2021, 1, 1)
end = datetime(2022, 12, 31)

all_data = []
current = start

while current <= end:
    next_date = current + timedelta(days=30)   # ⭐ 핵심 수정

    print(f"{current.date()} ~ {next_date.date()}")

    # 첫 페이지
    items, total = get_data(
        current.strftime("%Y%m%d"),
        next_date.strftime("%Y%m%d"),
        1
    )

    all_data.extend(items)

    page_size = 500
    total_pages = (total // page_size) + 1

    for page in range(2, total_pages + 1):
        items, _ = get_data(
            current.strftime("%Y%m%d"),
            next_date.strftime("%Y%m%d"),
            page
        )
        all_data.extend(items)

    current = next_date

# ----------------------
# 결과
# ----------------------
df = pd.DataFrame(all_data)

print("총 데이터 개수:", len(df))
print(df.head())

df.to_csv("raw_race_2021_to_2022.csv", index=False)

In [4]:
import pandas as pd

# 1. 로드
df1 = pd.read_csv(r"C:\Users\qkrwl\OneDrive\문서\GitHub\Horse\data_preprocessing\cleaned_data_1.csv")
df2 = pd.read_csv(r"C:\Users\qkrwl\OneDrive\문서\GitHub\Horse\data_raw\현역경주마_20260426.csv")
df3 = pd.read_csv(r"C:\Users\qkrwl\OneDrive\문서\GitHub\Horse\data_raw\horse_info.csv")

df1

C:\Users\qkrwl\AppData\Local\Temp\ipykernel_9316\2378418455.py:4: DtypeWarning: Columns (0: jkNo) have mixed types. Specify dtype option on import or set low_memory=False.
  df1 = pd.read_csv(r"C:\Users\qkrwl\OneDrive\문서\GitHub\Horse\data_preprocessing\cleaned_data_1.csv")


,chaksun,diffTot,divide,hrName,hrno,jkName,jkNo,noracefl,prow,prowName,...,rcSex,rcSpcbu,rcTime,rcVtdusu,rundayth,track,weath,wgHr,test_rank,rank
0,2000000,12.250,0,새내칸,45818,이혁,080486,정상,117013.0,문금철,...,오픈,1,83.3,12.0,1,양호,흐림,489.0,4.0,4.0
1,0,15.000,0,한강파워,45568,유승완,080417,정상,121008.0,나기두,...,오픈,1,83.8,12.0,1,양호,흐림,519.0,6.0,6.0
2,1600000,13.250,0,파이널축제,45339,함완식,080342,정상,113110.0,정기환,...,오픈,1,83.5,12.0,1,양호,흐림,490.0,5.0,5.0
3,0,21.375,0,런던비상,45409,박현우,080499,정상,121003.0,임현성,...,오픈,1,85.0,12.0,1,양호,흐림,446.0,12.0,12.0
4,0,17.500,0,나올탱크,46372,문성혁,080587,정상,121010.0,이경호a,...,오픈,1,84.2,12.0,1,양호,흐림,532.0,9.0,9.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33156,60500000,NaN,0,강풍마,45769,조재로,80550,정상,10496.0,박남성,...,오픈,0,127.8,11.0,98,다습,맑음,573.0,1.0,1.0
33157,24200000,6.000,0,원평스톰,55888,마이아,80616,정상,106198.0,김용재,...,오픈,0,128.9,11.0,98,다습,맑음,557.0,2.0,2.0
33158,5500000,6.625,0,블랙벨트,46935,정정희,80530,정상,121047.0,이강운,...,오픈,0,129.0,11.0,98,다습,맑음,506.0,4.0,4.0
33159,0,23.125,0,새내타운,45229,송재철,80513,정상,122027.0,문종원,...,오픈,0,131.8,11.0,98,다습,맑음,494.0,11.0,11.0


In [ ]:
# 3. merge
df = pd.merge(df1, df3, on="horse_id", how="left")
df = pd.merge(df, df2, on="horse_id", how="left")

# 4. 결측치 처리
df["diffTot"] = df["diffTot"].fillna(0)
df["rcAge"] = df["rcAge"].fillna(df["rcAge"].median())

# 5. 저장
df.to_csv("final_dataset.csv", index=False)